# Practice 4 — SQLite, HTTP, CLI, and Performance

SQLite, HTTP clients, pagination and argparse.

**How it works:** each exercise is almost-finished code with a few `___` blanks. Fill them in, run the cell, then run the `check(...)` cell below it. ✅ means move on. ❌ shows your result and the test input, never the answer, and failing twice unlocks a hint.

Prefer a terminal? The same exercises are in [`practice/exercises/intermediate_04_sql_http_cli.py`](../../practice/exercises/intermediate_04_sql_http_cli.py).

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "exercises"))
    sys.path.insert(0, os.getcwd())

## Setup

Run this cell first.

In [ ]:
from check import Checker, ___  # ___ marks the blanks you fill in

check = Checker('intermediate_04_sql_http_cli')

import argparse
import json
import sqlite3
import urllib.request
from collections.abc import Callable

## Exercises

### 1. `create_schema`

Create tables:

*Checked together with a later exercise that uses it.*

In [ ]:
def create_schema(conn: sqlite3.Connection) -> None:
    """Create tables:
    tasks(id INTEGER PRIMARY KEY, title TEXT NOT NULL CHECK(trim(title) != ''),
          done INTEGER NOT NULL DEFAULT 0)
    accounts(name TEXT PRIMARY KEY, balance REAL NOT NULL CHECK(balance >= 0))
    """
    ___(
        """
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL CHECK (trim(title) != ''),
            done INTEGER NOT NULL DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS accounts (
            name TEXT PRIMARY KEY,
            balance REAL NOT NULL CHECK (balance >= 0)
        );
        """
    )

### 2. `add_task`

Insert a task using a ? placeholder (never string formatting) and return its id.

In [ ]:
def add_task(conn: sqlite3.Connection, title: str) -> int:
    """Insert a task using a ? placeholder (never string formatting) and return its id."""
    with conn:
        cursor = ___
    return cursor.lastrowid

In [ ]:
check('add_task')

### 3. `list_tasks`

Return (id, title, done) tuples ordered by id; filter by done when it is not None.

In [ ]:
def list_tasks(conn: sqlite3.Connection, done=None) -> list[tuple[int, str, bool]]:
    """Return (id, title, done) tuples ordered by id; filter by done when it is not None."""
    sql, params = "SELECT id, title, done FROM tasks", ()
    if ___:
        sql, params = sql + " WHERE done = ?", (int(done),)
    return ___

In [ ]:
check('list_tasks')

### 4. `complete_task`

Mark a task done. Return False if no task has that id.

*Checked together with a later exercise that uses it.*

In [ ]:
def complete_task(conn: sqlite3.Connection, task_id: int) -> bool:
    """Mark a task done. Return False if no task has that id."""
    with conn:
        return ___

### 5. `delete_task`

Delete a task. Return False if no task has that id.

In [ ]:
def delete_task(conn: sqlite3.Connection, task_id: int) -> bool:
    """Delete a task. Return False if no task has that id."""
    with conn:
        return ___

In [ ]:
check('delete_task')

### 6. `transfer`

Move money between accounts atomically.

In [ ]:
def transfer(conn: sqlite3.Connection, source: str, target: str, amount: float) -> None:
    """Move money between accounts atomically.

    If the CHECK constraint fails (overdraft) or an account is missing, raise
    ValueError and leave BOTH balances unchanged. Hint: `with conn:` commits or rolls back.
    """
    try:
        with conn:
            for name, delta in ___:
                updated = conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?",
                                       (delta, name)).rowcount
                if ___:
                    raise ValueError(f"unknown account: {name}")
    except sqlite3.IntegrityError as error:
        raise ValueError("insufficient funds") from error

In [ ]:
check('transfer')

### 7. `count_by_status`

Return {'open': n, 'done': m} using one GROUP BY query.

In [ ]:
def count_by_status(conn: sqlite3.Connection) -> dict[str, int]:
    """Return {'open': n, 'done': m} using one GROUP BY query."""
    counts = ___
    for done, total in ___:
        counts["done" if done else "open"] = total
    return counts

In [ ]:
check('count_by_status')

### 8. `build_request`

Build a GET Request with headers Accept: application/json and User-Agent: course-client/1.0.

In [ ]:
def build_request(url: str) -> urllib.request.Request:
    """Build a GET Request with headers Accept: application/json and User-Agent: course-client/1.0."""
    return ___

In [ ]:
check('build_request')

### 9. `fetch_json`

Fetch and decode JSON. `opener(request, timeout=...)` returns an object with .read().

In [ ]:
def fetch_json(url: str, opener: Callable = urllib.request.urlopen, timeout: float = 5.0):
    """Fetch and decode JSON. `opener(request, timeout=...)` returns an object with .read().

    Raise ValueError if the body is not valid JSON. The opener is injected so
    tests never touch the network.
    """
    with opener(build_request(url), timeout=timeout) as response:
        body = ___
    try:
        return ___
    except json.JSONDecodeError as error:
        raise ValueError("response was not valid JSON") from error

In [ ]:
check('fetch_json')

### 10. `collect_pages`

Follow {'items': [...], 'next': <page or None>} pages and return all items.

In [ ]:
def collect_pages(fetch_page: Callable[[str], dict], first: str, max_pages: int = 100) -> list:
    """Follow {'items': [...], 'next': <page or None>} pages and return all items.

    Raise RuntimeError if more than max_pages pages are visited (protects against loops).
    """
    items, page, visited = ___
    while ___:
        visited += 1
        if ___:
            raise RuntimeError("too many pages")
        data = fetch_page(page)
        items.extend(data["items"])
        page = data.get("next")
    return items

In [ ]:
check('collect_pages')

### 11. `build_parser`

CLI: `tasks add TITLE` and `tasks list [--done]`; a subcommand is required.

In [ ]:
def build_parser() -> argparse.ArgumentParser:
    """CLI: `tasks add TITLE` and `tasks list [--done]`; a subcommand is required."""
    parser = ___
    sub = ___
    add = sub.add_parser("add", help="add a task")
    add.add_argument("title")
    listing = sub.add_parser("list", help="list tasks")
    listing.add_argument("--done", action="store_true", help="only show finished tasks")
    return parser

In [ ]:
check('build_parser')

### 12. `search_tasks`

Titles containing text (case-insensitive), ordered by id. Use LIKE with a ? parameter.

In [ ]:
def search_tasks(conn: sqlite3.Connection, text: str) -> list[str]:
    """Titles containing text (case-insensitive), ordered by id. Use LIKE with a ? parameter."""
    rows = conn.execute("SELECT title FROM tasks WHERE title LIKE ? ORDER BY id", (f"%{text}%",))
    return ___

In [ ]:
check('search_tasks')

### 13. `set_setting`

Insert or update a row in settings(key TEXT PRIMARY KEY, value TEXT) — an 'upsert'.

In [ ]:
def set_setting(conn: sqlite3.Connection, key: str, value: str) -> None:
    """Insert or update a row in settings(key TEXT PRIMARY KEY, value TEXT) — an 'upsert'.

    Hint: INSERT ... ON CONFLICT(key) DO UPDATE SET value = excluded.value
    """
    with conn:
        conn.execute("CREATE TABLE IF NOT EXISTS settings (key TEXT PRIMARY KEY, value TEXT)")
        conn.execute("INSERT INTO settings (key, value) VALUES (?, ?) "
                     "ON CONFLICT(key) DO UPDATE SET value = excluded.value", ___)

In [ ]:
check('set_setting')

### 14. `build_url`

Join base and path with exactly one '/', then add an encoded query string.

In [ ]:
def build_url(base: str, path: str, params: dict | None = None) -> str:
    """Join base and path with exactly one '/', then add an encoded query string.

    build_url('https://api.x/', '/search', {'q': 'a b'}) -> 'https://api.x/search?q=a+b'
    Hint: urllib.parse.urlencode
    """
    import urllib.parse
    url = base.rstrip("/") + "/" + path.lstrip("/")
    return ___

In [ ]:
check('build_url')

### 15. `paginate`

Return {'items': page slice, 'page': page, 'pages': total pages}. Pages start at 1.

In [ ]:
def paginate(items: list, page: int, per_page: int) -> dict:
    """Return {'items': page slice, 'page': page, 'pages': total pages}. Pages start at 1.
    Raise ValueError for page < 1 or per_page < 1. An empty list has 0 pages."""
    if ___:
        raise ValueError("page and per_page must be at least 1")
    start = (page - 1) * per_page
    return ___

In [ ]:
check('paginate')

## Finished?

In [ ]:
check.summary()